# Lesson 04 - টুল ব্যবহার ডিজাইন প্যাটার্ন

এই পাঠে আপনি Microsoft Agent Framework (Python) ব্যবহার করে AI এজেন্টদের জন্য **টুল ব্যবহার** ডিজাইন প্যাটার্ন শিখবেন। আমরা আলোচনা করব:

- `@tool` ডেকোরেটর এবং টাইপযুক্ত প্যারামিটার সহ ফাংশন টুল নির্ধারণ করা
- টুল স্কিমা প্রদান করা যাতে মডেল জানে প্রতিটি টুল কি করে
- `approval_mode` দিয়ে টুল কার্যক্রম নিয়ন্ত্রণ করা
- Pydantic মডেল এবং `response_format` এর মাধ্যমে **গঠিত আউটপুট** ফেরত দেওয়া

পরিসংখ্যানটি একটি **ভ্রমণ বুকিং এজেন্ট** যা গন্তব্য অনুসন্ধান করতে, উপলব্ধতা পরীক্ষা করতে, এবং ফ্লাইট তথ্য আহরণ করতে পারে।


## সেটআপ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ডেকোরেটরের মাধ্যমে টুল সংজ্ঞায়িত করা

`@tool` ডেকোরেটর একটি সাধারণ পাইথন ফাংশনকে এমন একটি টুলে রূপান্তর করে যা একটি এজেন্ট কল করতে পারে।
মূল পয়েন্টগুলি:

- **ডকস্ট্রিং** টুলের বর্ণনায় পরিণত হয় যা মডেল দেখে।
- **টাইপ অ্যানোটেশন** (বর্ণনাসহ `Annotated` সহ) টুলের স্কিমা নির্ধারণ করে।
- `approval_mode` নিয়ন্ত্রণ করে যে ব্যবহারকারীকে প্রতিটি কল কার্যকর করার আগে অনুমোদন দিতে হবে কিনা।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## একাধিক টুল সহ এজেন্ট তৈরি করা

모ডেলটি ব্যবহারকারীর প্রশ্নের উত্তর দেওয়ার জন্য যেকোনো টুল প্রয়োজন হলে সেইগুলি আহ্বান করতে পারে যাতে তিনটি টুলই ক্লায়েন্টকে পাঠান।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## সরঞ্জাম সম্বলিত কাঠামোবদ্ধ আউটপুট

`response_format` একটি Pydantic মডেলে সেট করার মাধ্যমে, এজেন্টকে নিখুঁত টাইপযুক্ত JSON অবজেক্ট ফেরত দিতে বাধ্য করা হয়, মুক্ত ফর্মের পাঠ্যের বদলে। যখন নিচের ধাপের কোড ফলাফল প্রোগ্রামগতভাবে গ্রহণ করতে হয় তখন এটি লাভজনক হয়।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## টুল অনুমোদন প্যাটার্ন

`@tool` এ `approval_mode` প্যারামিটারটি নিয়ন্ত্রণ করে টুল কলগুলি কার্যকর করার আগে মানব অনুমোদন দরকার কিনা:

| মোড | আচরণ |
|---|---|
| `"never_require"` | টুল স্বয়ংক্রিয়ভাবে চলে — কোনো ব্যবহারকারী নিশ্চিতকরণ দরকার নেই। |
| `"always_require"` | প্রতিটি কল কার্যকর করার পূর্বে ব্যবহারকারীর দ্বারা অনুমোদিত হতে হবে। |

যেসব টুলের পার্শ্বপ্রতিক্রিয়া থাকে (যেমন একটি ফ্লাইট বুকিং করা, ক্রেডিট কার্ডে চার্জ করা) সেগুলির জন্য `"always_require"` ব্যবহার করুন যাতে মানব ব্যক্তি প্রক্রিয়ায় থাকে।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## সারাংশ

এই পাঠে আপনি শিখেছেন কিভাবে:

1. **টুল সংজ্ঞায়িত করবেন** `@tool` ডেকোরেটর ব্যবহার করে, যার টাইপ করা প্যারামিটার এবং ডকস্ট্রিং থাকে যা টুলের স্কিমা হিসেবে কাজ করে।
2. **একাধিক টুল তৈরি করবেন** যাতে এজেন্ট এগুলো ধারাবাহিকভাবে কল করতে পারে জটিল প্রশ্নের উত্তর দিতে।
3. **গঠনমূলক আউটপুট প্রদান করবেন** একটি Pydantic মডেল `response_format` হিসেবে পাস করে।
4. **টুল অনুমোদন নিয়ন্ত্রণ করবেন** `approval_mode` দিয়ে যাতে সংবেদনশীল অপারেশনে মানুষের অনুমতি অন্তর্ভুক্ত করা যায়।

এই প্যাটার্নগুলো বিশ্বাসযোগ্য, প্রস্তুত প্রোডাকশন এজেন্ট তৈরি করার ভিত্তি গঠন করে যারা বাইরের সিস্টেমের সঙ্গে নিরাপদে ইন্টারঅ্যাক্ট করতে পারে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ব্যক্তিগত আপত্তি**:
এই দলিলটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা যথাসম্ভব সঠিকতার চেষ্টা করি, স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথি তার আদিম ভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ গ্রহণ করার পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারের ফলে কোনো ভুল বোঝাবুঝি বা ব্যাখ্যার জন্য আমরা কোন ধরনের দায় স্বীকার করি না।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
